In [2]:
import logging
import os

from dotenv import load_dotenv
from matplotlib.ticker import FuncFormatter
from matplotlib import pyplot as plt
from pathlib import Path
from typing import Tuple

import numpy as np
import polars as pl

# Env
pl.Config.set_tbl_rows(10)
load_dotenv("../.env")
INSTALL_PATH = os.environ["INSTALL_PATH"]
AF_URL = os.environ["API_URL_ALPHAFOLD_STRUCTPRED"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

### Steps

1. Apply preliminary filtering criteria
2. Apply PyMOL alignment
3. Lookup how SMARCA4, KPNA2 fare

In [10]:
# ────────────────────────────────────────────────────────────
#      Args
# ────────────────────────────────────────────────────────────

THRESHOLD_LENGTH_ERROR = 0.05,
THRESHOLD_OVERLAP_RATIO = 0.9
PYMOL_ALIGN_METHOD = "super"

OUT_DIR_AF = Path(f"{INSTALL_PATH}/data_local/alphafold_structures")
OUT_DIR_PM = Path(f"{INSTALL_PATH}/data_local/pymol_alignments/{PYMOL_ALIGN_METHOD}")

data_file = f"{INSTALL_PATH}/data_local/zorya/2025-07-27_zorya_annotated.tsv"

In [12]:
# ────────────────────────────────────────────────────────────
#      In
# ────────────────────────────────────────────────────────────

OUT_DIR_AF.mkdir(exist_ok=True, parents=True)
OUT_DIR_PM.mkdir(exist_ok=True, parents=True)

df = pl.read_csv(
            data_file,
            has_header=True,
            separator="\t",
            schema_overrides={
                "duplicate_count": pl.Int64,
                "rank": pl.Int64,
                "evalue": pl.Float64,
                "cluFlag": pl.Int64,
                "fident": pl.Float64,
                "alnlen": pl.Int64,
                "mismatch": pl.Int64,
                "gapopen": pl.Float64,
                "qstart": pl.Int64,
                "qend": pl.Int64,
                "tstart": pl.Int64,
                "tend": pl.Int64,
                "evalue_foldseek": pl.Float64,
                "bits": pl.Int64,
                "Query_length": pl.Int64,
                "Human_prot_total_length": pl.Int64,
                "Human_domain_percentage": pl.Float64,
                "Query_percentage": pl.Float64,
                "Query_overlap_w_Human_protein": pl.Float64,
                "Target_length": pl.Int64,
                "Bacterial_prot_total_length": pl.Int64,
                "Bacterial_domain_percentage": pl.Float64,
                "Target_percentage": pl.Float64,
                "Target_overlap_w_Bacterial_protein": pl.Float64,
                "Bacterial_Human_Len_Ratio": pl.Float64,
                "Query_Target_Overlap_Length": pl.Float64,
                "Overlap_ratio": pl.Float64,
            },
            infer_schema_length=10000
        )

df

accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64
"""WP_063192462.1""","""Zorya""","""Zorya_TypeI""","""Geobacillus sp. JS12""","""GCF_001592395_NZ_CP014749_Zory…","""GCF_001592395.1_NZ_CP014749_00…","""GCF_001592395.1_NZ_CP014749_00…",1,202,"""A0A142CZW3""","""A0A355UP19""","""A0A355UP19""","""B4DLD8""",0.0,1,0.21,662,519,0.0,277,934,202,863,3.9580e-33,807,"""unreviewed""","""B4DLD8_HUMAN""","""DNA helicase (EC 3.6.4.12)""",null,"""Homo sapiens (Human)""",null,null,null,null,null,"""chromatin organization [GO:000…","""nucleus [GO:0005634]""","""nucleus [GO:0005634]; ATP bind…","""ATP binding [GO:0005524]; DNA …","""GO:0003677; GO:0004386; GO:000…","""cd18668; CD1_tandem_CHD5-9_lik…","""2.40.50.40:FF:000001; chromodo…","""2.40.50.40; -; 2.;""3.40.50.300…","""IPR051493; CHD.;""IPR016197; Ch…","""PTHR46850; CHROMODOMAIN-HELICA…","""PF00385; Chromo; 2.;""PF00271; …","""PS50013; CHROMO_2; 1.;""PS00690…","""SM00298; CHROMO; 2.;""SM00487; …",null,"""277.0..934.0""",658,1014,"""353..396""",100.0,6.69,64.89,"""['Chromo']""","""['ECO:0000259|PROSITE:PS50013'…","""202.0..863.0""",662,874,"""698..851""",100.0,23.26,75.74,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…",0.861933,658.0,0.75286
"""WP_063192462.1""","""Zorya""","""Zorya_TypeI""","""Geobacillus sp. JS12""","""GCF_001592395_NZ_CP014749_Zory…","""GCF_001592395.1_NZ_CP014749_00…","""GCF_001592395.1_NZ_CP014749_00…",1,202,"""A0A142CZW3""","""A0A355UP19""","""A0A355UP19""","""B4DLD8""",0.0,1,0.21,662,519,0.0,277,934,202,863,3.9580e-33,807,"""unreviewed""","""B4DLD8_HUMAN""","""DNA helicase (EC 3.6.4.12)""",null,"""Homo sapiens (Human)""",null,null,null,null,null,"""chromatin organization [GO:000…","""nucleus [GO:0005634]""","""nucleus [GO:0005634]; ATP bind…","""ATP binding [GO:0005524]; DNA …","""GO:0003677; GO:0004386; GO:000…","""cd18668; CD1_tandem_CHD5-9_lik…","""2.40.50.40:FF:000001; chromodo…","""2.40.50.40; -; 2.;""3.40.50.300…","""IPR051493; CHD.;""IPR016197; Ch…","""PTHR46850; CHROMODOMAIN-HELICA…","""PF00385; Chromo; 2.;""PF00271; …","""PS50013; CHROMO_2; 1.;""PS00690…","""SM00298; CHROMO; 2.;""SM00487; …",null,"""277.0..934.0""",658,1014,"""353..396""",100.0,6.69,64.89,"""['Chromo']""","""['ECO:0000259|PROSITE:PS50013'…","""202.0..863.0""",662,874,"""698..851""",100.0,23.26,75.74,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…",0.861933,658.0,0.75286
"""WP_063192462.1""","""Zorya""","""Zorya_TypeI""","""Geobacillus sp. JS12""","""GCF_001592395_NZ_CP014749_Zory…","""GCF_001592395.1_NZ_CP014749_00…","""GCF_001592395.1_NZ_CP014749_00…",1,202,"""A0A142CZW3""","""A0A355UP19""","""A0A355UP19""","""B4DLD8""",0.0,1,0.21,662,519,0.0,277,934,202,863,3.9580e-33,807,"""unreviewe

In [9]:
# ────────────────────────────────────────────────────────────
#      Clean
# ────────────────────────────────────────────────────────────

def remove_dup_rows(df: pl.DataFrame) -> tuple[pl.DataFrame, int]:
    df_dup_counted = (
        df.group_by(df.columns)
        .len()
    )
    dups_count = (
        df_dup_counted.filter(pl.col("len") > 1)
        .select((pl.col("len") - 1).sum())
        .item()
    )
    return df_dup_counted.drop("len"), dups_count

df_cl, _ = remove_dup_rows(df)
logger.info(f"N rows removed: {_}")

df_cl

INFO:__main__:N rows removed: 40578


accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64
"""WP_151184585.1""","""Zorya""","""Zorya_TypeI""","""Pseudomonas sp. C27(2019)""","""GCF_008807395_NZ_CP043320_Zory…","""GCF_008807395.1_NZ_CP043320_01…","""GCF_008807395.1_NZ_CP043320_01…",1,202,"""A0A5J6QBU2""","""A0A355UP19""","""A0A2D7ICJ8""","""A0A087X072""",6.9410e-10,2,0.118,591,391,0.0,19,462,391,981,3.2080e-9,183,"""unreviewed""","""A0A087X072_HUMAN""","""DNA 3'-5' helicase (EC 5.6.2.4…","""RECQL4""","""Homo sapiens (Human)""",null,null,"""RECQL4""",null,"""UP000005640: Chromosome 8""","""DNA recombination [GO:0006310]…",null,"""ATP binding [GO:0005524]; heli…","""ATP binding [GO:0005524]; heli…","""GO:0003676; GO:0004386; GO:000…","""cd18018; DEXHc_RecQ4-like; 1.;…","""3.40.50.300:FF:000772; ATP-dep…","""3.40.50.300; P-loop containing…","""IPR011545; DEAD/DEAH_box_helic…","""PTHR13710:SF108; ATP-DEPENDENT…","""PF00270; DEAD; 1.;""PF00271; He…","""PS51192; HELICASE_ATP_BIND_1; …","""SM00487; DEXDc; 1.;""SM00490; H…","""NP_001399950.1; NM_001413021.1…","""19.0..462.0""",444,851,"""132..305""",100.0,39.19,52.17,"""['Helicase ATP-binding']""","""['ECO:0000259|PROSITE:PS51192'…","""391.0..981.0""",591,1024,"""489..698""",100.0,35.53,57.71,"""['Helicase ATP-binding']""","""['ECO:0000259|PROSITE:PS51192'…",1.20329,444.0,0.521739
"""WP_128105540.1""","""Zorya""","""Zorya_TypeI""","""Acetobacter oryzoeni""","""GCF_004014775_NZ_CP042808_Zory…","""GCF_004014775.2_NZ_CP042808_01…","""GCF_004014775.2_NZ_CP042808_01…",1,202,"""A0A5B9GHV7""","""A0A355UP19""","""A0A0D0B8K2""","""Q4W5H1""",7.8290e-19,3,0.248,344,258,0.0,20,363,627,970,8.4210e-22,658,"""unreviewed""","""Q4W5H1_HUMAN""","""Uncharacterized protein SMARCA…","""SMARCA5""","""Homo sapiens (Human)""",null,null,"""SMARCA5""",null,null,"""chromatin remodeling [GO:00063…","""nucleus [GO:0005634]""","""nucleus [GO:0005634]; ATP bind…","""ATP binding [GO:0005524]; hydr…","""GO:0005524; GO:0005634; GO:000…","""cd18793; SF2_C_SNF; 1.;""","""3.40.50.10810:FF:000192; Chrom…","""3.40.50.300; P-loop containing…","""IPR014001; Helicase_ATP-bd.;""I…","""PTHR45623; CHROMODOMAIN-HELICA…","""PF00271; Helicase_C; 1.;""PF001…","""PS51192; HELICASE_ATP_BIND_1; …","""SM00490; HELICc; 1.;""",null,"""20.0..363.0""",344,367,"""220..367""",97.3,41.86,93.73,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…","""627.0..970.0""",344,1028,"""505..697""",36.79,20.64,33.46,"""['Helicase ATP-binding']""","""['ECO:0000259|PROSITE:PS51192'…",2.80109,344.0,0.9373297
"""WP_138224688.1""","""Zorya""","""Zorya_TypeI""","""Paenibacillus algicola""","""GCF_005577435_NZ_CP040396_Zory…","""GCF_005577435.1_NZ_CP040396_00…","""GCF_005577435.1_NZ_CP040396_00…",1,202,"""A0A4P8XGJ0""","